# QLoRA

QLoRA 是一种省显存的微调方法, 其核心是通过量化减少参数的显存占用, 使得在单机内训练超大的模型.

QLoRA 实际上是 “量化” 技术展开的, 而非是 PEFT 方法.

QLoRA 精度一般, 该方法较为鸡肋, 本 lecture 不深入讨论 QLoRA 的量化细节, 仅代码实现其计算过程.

# Transformers 调用 QLoRA

量化: https://huggingface.co/docs/peft/developer_guides/quantization#quantize-a-model

LoRA: https://huggingface.co/docs/peft/developer_guides/quantization#loraconfig

In [1]:
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# import torch

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )

# tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
# model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B', quantization_config=bnb_config) 


# # tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B', 
# #                                           local_dir='~/.cache/huggingface/', # 如果可以直连 huggingface, 去除此行.
# #                                          )

# # model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B',
# #                                              local_files_only=True, 
# #                                             # quantization_config=bnb_config
# #                                             ) 

# # 观测显存 1B(fp16) = 2G, 对于 4bit, 约 0.5G 显存占用

In [2]:
# from peft import LoraConfig
# from peft import get_peft_model
# config = LoraConfig(
#     r=16,
#     lora_alpha=8,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM"
# )
# model = get_peft_model(model, config)
# print(model)

In [3]:
# from trl import SFTTrainer
# from datasets import load_dataset
# from  trl import SFTConfig #https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig

# config = SFTConfig(
#     output_dir="output/qwen3_sft",
#     per_device_train_batch_size = 2,
#     max_length = 256,
#     max_steps = 10
# )

# trainer = SFTTrainer(
#     model=model,
#     args=config,
#     train_dataset=load_dataset("trl-lib/Capybara", split="train"),
# )
# trainer.train()

# QLoRA 实现

QLoRA 实现上仅在 LoRA 基础上对原参数层进行量化, 在计算过程中, 动态去量化(de-quant).

为了简化实现, 我们仅对模型所出现的 Linear 层处理, 如 proj, lm_heads

1. 将 fp16 线性层替换为 int4(可自选), `QuantizedLinear`
2. 将 `QuantizedLinear` 增加 LoRA 参数, 得到 `QuantizedLoRALinear`
3. `QuantizedLoRALinear` 前向时主分支计算需要禁用梯度

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

dim = 512
r = 4
bsz = 1
seq_len = 128
vocab_size = 100

x = torch.randint(vocab_size, (bsz, seq_len))
X = torch.randn(bsz, seq_len, dim)

NameError: name 'torch' is not defined

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.Wq = nn.Linear(dim_in, dim_out, bias=False)
        self.Wk = nn.Linear(dim_in, dim_out, bias=False)
        self.Wv = nn.Linear(dim_in, dim_out, bias=False)
        self.Wo = nn.Linear(dim_out, dim_out, bias=False)
    def forward(self, X):
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        S = Q@K.transpose(1,2)
        P = F.softmax(S, dim = -1)
        Z = P @ V
        O = self.Wo(Z)
        return O

model = Attention(dim, dim)
model(X)
print(model)

## INT8 Linear

In [ ]:
import bitsandbytes as bnb

class QuantizedLinear(nn.Module):
    def __init__(self, module, dim_in, dim_out, block_size=64):
        super().__init__()

        self.in_features = dim_in
        self.out_features = dim_out
        self.block_size = block_size
        
        # 原始精度
        self.weight = nn.Parameter(module.weight.data)
        if module.bias:
            self.bias = nn.Parameter(module.bias)
        else:
            self.register_parameter('bias', None)
        
        self.register_buffer('quant_weight', None)
        # self.register_buffer('weight_scale', None)
        self.weight_scale = None
        self.quantized = False

        self.quantize_()

    def quantize_(self, block_size=64):
        w = self.weight.data
        quant_weight, quant_state = bnb.functional.quantize_blockwise(w, 
                                                                      blocksize=self.block_size)
        print(quant_state)
        self.quant_weight = quant_weight  # int8 类型
        self.weight_scale = quant_state
        self.weight = None # 删除参数减少显存
        self.quantized = True

    def forward(self, x):
        if not self.quantized:
            return nn.functional.linear(x, self.weight, self.bias)
        else:
            # 计算时仍然用 fp16
            dequant_weight = bnb.functional.dequantize_blockwise(
                self.quant_weight, 
                self.weight_scale, 
                blocksize=self.block_size
            )
            return nn.functional.linear(x, dequant_weight, self.bias)

In [ ]:
def apply_quant(model, MyLinear, dim, block_size):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            new_layer = MyLinear(module, dim, dim, block_size)
            setattr(model, name, new_layer)
            print(f"Replaced {name}")
        else:
            # 递归替换
            apply_quant(module, QuantizedLinear, dim, block_size)
    return model

block_size = 64
apply_quant(model, QuantizedLinear, dim, block_size)
print(model)
with torch.no_grad():
    y = model(X)
    print(y)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, original_linear, rank=4, alpha=0.1):
        super().__init__()
        self.alpha = alpha
        self.dim_in = original_linear.in_features
        self.dim_out = original_linear.out_features
        self.r=rank

        # 原始参数
        self.weight = original_linear

        # LoRA参数
        self.WA = nn.Linear(self.dim_in, self.r, bias=False)
        self.WB = nn.Linear(self.r, self.dim_out, bias=False)

    def forward(self, X):
        with torch.no_grad():
            h = self.weight(X)
        h_lora = h + self.alpha * self.WB(self.WA(X))
        return h_lora 

In [ ]:
def apply_lora_adapter(model, MyLinear, r):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear): # LoRA
            new_layer = MyLinear(module, r)
            setattr(model, name, new_layer)
            print(f"Replaced {name}")
        elif isinstance(module, QuantizedLinear): # QLoRA
            new_layer = MyLinear(module, r)
            setattr(model, name, new_layer)
            print(f"Replaced {name}")
        else:
            # 递归替换
            apply_lora_adapter(module, MyLinear, r)
    return model

apply_lora_adapter(model, LoRALinear, r=4)
print(model)

## Train

In [ ]:
H = model(X)
label = torch.randn(bsz, seq_len, dim)
loss_fn = nn.MSELoss()
loss = loss_fn(H.view(-1), label.view(-1))
print(loss)
loss.backward()

In [ ]:
print(model.Wq.weight.quant_weight) # 无梯度, forward 后参数仍为 int8
print(model.Wq.WA.weight.grad) # 有梯度